# Qwen planting localization development

Select a GPU under **Runtime → Change runtime type**, then choose **Run all**.
The run compares four combinations of training scope and neighbor rehearsal
on development seeds 221, 222, and 223. It uses selection prompts and retains
per-checkpoint strength, spillover, collateral losses, and arithmetic scores.

[Read the procedure](https://github.com/AjayKasu1/intervention-ladder/blob/a45e87637aaf55ae60b7cbffa49f51b1c4d5be1b/docs/PLANT_LOCALIZATION.md).
Keep this runtime connected until the reports download.


In [ ]:
# @title 1. Load the pinned code and Python dependencies
import importlib.metadata
import json
import os
from pathlib import Path
import shutil
import subprocess
import sys
import tempfile
import zipfile

if shutil.which("nvidia-smi") is None:
    raise RuntimeError("Select a GPU runtime before running this notebook.")

CODE_REVISION = "a45e87637aaf55ae60b7cbffa49f51b1c4d5be1b"
REPO = Path("/content/intervention-ladder-localization-v3")
if not REPO.exists():
    subprocess.run(["git", "clone", "https://github.com/AjayKasu1/intervention-ladder.git", str(REPO)], check=True)
subprocess.run(["git", "fetch", "origin"], cwd=REPO, check=True)
subprocess.run(["git", "checkout", "--detach", CODE_REVISION], cwd=REPO, check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "transformers==4.57.6",
                "numpy>=1.26", "scipy>=1.11", "pyyaml>=6.0"], check=True)
import torch

if not torch.cuda.is_available():
    raise RuntimeError("PyTorch cannot access a CUDA device.")
print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
# @title 2. Run the fresh development comparison
WORK = Path(tempfile.mkdtemp(prefix="qwen-localization-v3-", dir="/content"))
OUT = WORK / "reports"
LOG = WORK / "development.log"
manifest = {
    "run_mode": "development", "excluded_from_pilot": True,
    "code_revision": CODE_REVISION, "gpu": torch.cuda.get_device_name(0),
    "python": sys.version.split()[0],
    "packages": {p: importlib.metadata.version(p) for p in
                 ("torch", "transformers", "numpy", "scipy", "pyyaml")},
}
(WORK / "manifest.json").write_text(json.dumps(manifest, indent=2))
with LOG.open("w") as log, subprocess.Popen(
    [sys.executable, "-u", "scripts/run_plant_development.py",
     "--model", "Qwen/Qwen2.5-0.5B-Instruct", "--experiment", "configs/plant_localization.yaml",
     "--out", str(OUT), "--progress-every", "25"],
    cwd=REPO, env={**os.environ, "HF_HUB_DISABLE_IMPLICIT_TOKEN": "1"},
    text=True, bufsize=1, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
) as process:
    for line in process.stdout:
        print(line, end="", flush=True)
        log.write(line)
        log.flush()
    exit_status = process.wait()
print("Development process exit status:", exit_status)


In [ ]:
# @title 3. Download the development reports
from google.colab import files

bundle = WORK / "qwen_plant_localization_reports.zip"
with zipfile.ZipFile(bundle, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for item in [WORK / "manifest.json", LOG, *sorted(OUT.glob("*.json*"))]:
        if item.is_file():
            archive.write(item, str(item.relative_to(WORK)))
files.download(str(bundle))
